# Optimisation du preproc

In [169]:
import pandas as pd
data = pd.read_csv("../../raw_data/recipes_ingredients.csv")

In [170]:
data.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


## On divise la colonne serving_size en 2

In [171]:
data[["persons", "portion_size"]] = data["serving_size"].str.extract(
    r"(\d+)\s*\(([^)]+)\)")
data = data.drop(columns="serving_size", axis=1)

on garde seulement les recettes avec moins de 50 portions et on enlève les colonnes inutiles

In [172]:
data = data[data["servings"] <= 50]

In [173]:
data = data.drop(["id","description"], axis=1)

## on passe les strings en listes

In [174]:
import ast
def safe_literal_eval(value):
    """
    Transforme les strings en liste de string
    """

    try:
        result = ast.literal_eval(value)

        if isinstance(result, list):
            return result

        return []

    except (ValueError, SyntaxError, TypeError):
        return []

on en profite pour garder les recettes pertinentes

In [175]:
def transform_str_to_list(data):
    data["ingredients"] = data["ingredients"].apply(safe_literal_eval)
    data["ingredients_raw"] = data["ingredients_raw"].apply(safe_literal_eval)
    data["steps"] = data["steps"].apply(safe_literal_eval)
    data["tags"] = data["tags"].apply(safe_literal_eval)
    #Vire les listes vides
    data = data[data["ingredients"].apply(len) > 2]
    data = data[data["ingredients"].apply(len) < 15]
    data = data[data["ingredients_raw"].apply(len) > 2]
    data = data[data["ingredients_raw"].apply(len) < 15]
    data = data[data["steps"].apply(len) > 1]
    data = data[data["tags"].apply(len) > 0]
    return data

In [176]:
data = transform_str_to_list(data)

## Nouvelles colonnes pour gérer les tags

In [177]:
from mastershelf.recipes.preprocessing import *
data = data[data["tags"].apply(lambda tags: any(tag in tags for tag in TIME_TO_MAKE))]
data['type_dish'] = data['tags'].apply(lambda x: type_column(x, TYPE_DISH))
data['type_diet'] = data['tags'].apply(lambda x: type_column(x, TYPE_DIET))
data['type_meal'] = data['tags'].apply(lambda x: type_column(x, TYPE_MEAL))
data['type_occasion'] = data['tags'].apply(lambda x: type_column(x, TYPE_OCCASION))
data['type_origin'] = data['tags'].apply(lambda x: type_column(x, TYPE_ORIGIN))
data = data[data["tags"].apply(len) > 0]

In [178]:
data.shape

(249454, 13)

## On Clean la colonne ingrédients

In [179]:
from mastershelf.recipes.params import *
import re


def clean_ingredient(text):
    text = text.lower().strip()


    text = re.sub(r"\b\d+([./]\d+)?\b", " ", text)

    for word in WORDS_TO_REMOVE:
        text = re.sub(rf"\b{re.escape(word)}\b", " ", text)


    text = re.sub(r"[,()]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [180]:
data["ingredients_clean"] = data["ingredients"].apply(
    lambda ingredients: [
        clean_ingredient(x)
        for x in ingredients
    ]
)

In [181]:
data.shape

(249454, 14)

In [182]:
data.head()

,name,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size,type_dish,type_diet,type_meal,type_occasion,type_origin,ingredients_clean
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,"[60-minutes-or-less, time-to-make, course, mai...",1,347 g,[cobblers-and-crisps],[],[desserts],[],[north-american],"[cherry pie filling, condensed milk, margarine..."
1,Reuben and Swiss Casserole Bake,"[corned beef chopped, sauerkraut cold water, s...","[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,"[60-minutes-or-less, time-to-make, course, mai...",1,207 g,[],[dietary],[main-dish],[],[],"[corned beef, sauerkraut cold water, swiss che..."
3,Tropical Orange Layer Cake,"[orange cake mix, instant vanilla pudding, ora...","[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,"[60-minutes-or-less, time-to-make, course, pre...",1,191 g,[cakes],"[dietary, low-protein]",[desserts],[],[],"[orange cake mix, instant vanilla pudding, ora..."
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,"[15-minutes-or-less, time-to-make, course, mai...",1,26 g,[cookies-and-brownies],[],[desserts],[],[],"[butter, brown sugar, granulated sugar, milk, ..."
9,Granny's Butter Rolls,"[biscuit mix, granulated sugar, butter, milk, ...","[2 1/4 cups biscuit mix, 2/3 cup water...",[Mix biscuit mix and water until a soft dough ...,12.0,"[60-minutes-or-less, time-to-make, course, pre...",1,91 g,"[breads, rolls-biscuits]","[dietary, vegetarian, comfort-food]",[brunch],[brunch],[],"[biscuit mix, granulated sugar, butter, milk, ..."


nombres d'ingrédients uniques

In [183]:
unique_clean = (
    data["ingredients_clean"]
    .explode()
    .dropna()
    .unique()
)

len(unique_clean)

115391

On check les 4800 ingredients les plus communs (90% coverage du dataset)

In [184]:
ingredient_counts = (
    data["ingredients_clean"]
    .explode()
    .value_counts()
)
most_commun = ingredient_counts.head(4800).index.tolist()


export pour le mapping

In [79]:
with open('../../raw_data/your_file.txt', 'w') as f:
    for line in most_commun:
        f.write(f"{line}\n")

## On map cette liste d'ingrédients pour regrouper les mêmes ingrédients écris sous différentes formes

In [185]:
import json

with open("../../raw_data/mapping_v5.json", "r", encoding="utf-8") as f:
    ingredient_mapping = json.load(f)

In [186]:
data["ingredients_clean"] = data["ingredients_clean"].apply(
    lambda ingredients: [
        ingredient_mapping.get(ingredient, ingredient)
        for ingredient in ingredients
    ]
)

On prends les 200 ingrédients les plus présents
La valeur à été trouvée en regarder la partie du dataset gardée pour trouver un équilibre

In [187]:
ingredient_counts = (
    data["ingredients_clean"]
    .explode()
    .value_counts()
)
most_commun = ingredient_counts.head(200).index.tolist()

export pour les copaings

In [188]:
with open('../../raw_data/most_communV2.txt', 'w') as f:
    for line in most_commun:
        f.write(f"{line}\n")

In [189]:
cumulative = ingredient_counts.cumsum() / ingredient_counts.sum()

print("80% coverage:", (cumulative <= 0.80).sum())
print("90% coverage:", (cumulative <= 0.90).sum())
print("95% coverage:", (cumulative <= 0.95).sum())

80% coverage: 157
90% coverage: 2099
95% coverage: 24742


On récupère seulement les recettes qui utilisent des ingrédients présents dans les 200

In [190]:
most_commons_set = set(most_commun)

df_filtered = data[
    data["ingredients_clean"].apply(
        lambda ingredients: set(ingredients).issubset(most_commons_set)
    )
].copy()

In [191]:
print(len(data))
print(len(df_filtered))

print(
    f"{len(df_filtered) / len(data) * 100:.2f}% des recettes conservées"
)

249454
75856
30.41% des recettes conservées


# Petit check sur des lignes randoms pour voir la pertinence du mapping

In [219]:
import random

x = random.randint(0, 75000)
pd.DataFrame(df_filtered.iloc[x].ingredients_clean , df_filtered.iloc[x].ingredients)

,0
tomatoes,tomato
zucchini chopped,zucchini
celery chopped,celery
green bell pepper,bell pepper
onion,onion
lemon juice,lemon juice
dried basil,basil
cumin,cumin
lemon zest,lemon


In [220]:
new_unique_clean = (
    df_filtered["ingredients_clean"]
    .explode()
    .dropna()
    .unique()
)

len(new_unique_clean)

200

In [221]:
df_filtered.head()

,name,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size,type_dish,type_diet,type_meal,type_occasion,type_origin,ingredients_clean
13,Crepes for Two,"[milk, vegetable oil, powdered sugar, orange m...","[1/2 cup flour, 3/4 cup milk, 1 ...","[Combine flour, milk, eggs, and oil and salt.,...",2.0,"[15-minutes-or-less, time-to-make, course, cui...",1,243 g,[pancakes-and-waffles],[dietary],[breakfast],[],"[french, european]","[milk, cooking oil, powdered sugar, orange, sp..."
17,Peanut Butter and Banana Burrito,"[flour tortilla, peanut butter, banana]","[1 flour tortilla, 1 -2 tablespoon ...","[Heat tortilla over open flame of burner, till...",1.0,"[15-minutes-or-less, time-to-make, course, mai...",1,105 g,[],"[dietary, kid-friendly, inexpensive]","[brunch, breakfast, lunch]",[brunch],"[north-american, american, southern-united-sta...","[flour, peanut butter, banana]"
25,Oil and Vinegar Salad Dressing,"[olive oil, white wine vinegar, dry mustard]","[3/4 cup olive oil, 5 tablespoons whi...",[Combine all ingredients in a jar and shake or...,1.0,"[15-minutes-or-less, time-to-make, course, pre...",1,169 g,"[salads, salad-dressings]",[dietary],[],[],[],"[cooking oil, vinegar, mustard powder]"
40,Southwestern Avocado Salsa,"[avocados, tomatoes, white shoepeg corn, black...","[2 avocados, diced , 2 tomatoes, d...",[Combine shoepeg corn and black beans in a med...,4.0,"[15-minutes-or-less, time-to-make, course, mai...",1,326 g,[vegetables],"[dietary, low-sodium, vegetarian, low-choleste...",[appetizers],[],[],"[avocado, tomato, corn, black bean, green onio..."
59,Chickpea Salad,"[chickpeas, celery, red onion, parsley, lemon ...","[16 ounces chickpeas, drained and rinsed ...","[Mix all ingredients together., Allow five min...",6.0,"[15-minutes-or-less, time-to-make, course, mai...",1,99 g,"[salads, vegetables]","[dietary, vegetarian, pasta-rice-and-grains, k...",[side-dishes],[],"[asian, middle-eastern]","[chickpea, celery, onion, parsley, lemon juice..."


## On gère la colonne time to make

petite fonction de tri

In [222]:
def clean_time(x):
    for element in x:
        if element in TIME_TO_MAKE:
            return element
    return "No info"

In [223]:
df_filtered["Time_to_make"] = df_filtered["tags"].apply(clean_time)

In [224]:
df_filtered.sample(10)

,name,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size,type_dish,type_diet,type_meal,type_occasion,type_origin,ingredients_clean,Time_to_make
235972,Sue's Grilled Ham and Cheese (Croque Monsieur),"[sliced ham, soft cream cheese neufchatel chee...","[2 slices bread, 2 ounces thinly slic...",[Spread bread with however much mustard you li...,1.0,"[ham, 15-minutes-or-less, time-to-make, course...",1,150 g,[sandwiches],"[comfort-food, inexpensive]",[lunch],[],"[north-american, french, european, american, m...","[pork, cream cheese, mustard, butter]",15-minutes-or-less
151613,Creamy Spinach and Ham,"[spinach, soft cheese, nutmeg, ham]","[2 slices bread, 1 handful spinach, ...",[Brush the sandwich toaster with a little sunf...,1.0,"[15-minutes-or-less, time-to-make, course, mai...",1,50 g,[sandwiches],[],[lunch],[],[],"[spinach, cheese, nutmeg, pork]",15-minutes-or-less
437393,Balsamic Maple Carrot Coins,"[carrots, maple syrup, balsamic vinegar, fresh...","[3 1/4 cups carrots, peeled, sliced 1/4 in...","[Steam carrots, covered, 15 minutes or until t...",4.0,"[30-minutes-or-less, time-to-make, course, mai...",1,114 g,[vegetables],[],[side-dishes],[],[],"[carrot, maple syrup, vinegar, pepper, parsley]",30-minutes-or-less
342394,"Asparagus, Yellow Pepper &amp; Tomato Salad","[fresh asparagus, yellow pepper, grape tomatoe...","[1 lb fresh asparagus, trimmed & cut into...",[Put the asparagus pieces in a small strainer-...,4.0,"[15-minutes-or-less, time-to-make, course, mai...",1,136 g,"[salads, vegetables]",[],"[potluck, picnic, side-dishes]","[potluck, picnic, to-go, summer, spring]",[],"[asparagus, bell pepper, tomato, green onion, ...",15-minutes-or-less
465575,Sticky Toffee Pudding,"[dried chopped dates, water, baking soda, unsa...","[8 ounces dried chopped dates, 1 1/3 cups...","[Cake directions:, Preheat oven to 350 degrees...",1.0,"[60-minutes-or-less, time-to-make, course, pre...",1,2760 g,[cakes],"[dietary, low-protein]",[desserts],[],[],"[date, water, baking soda, butter, baking powd...",60-minutes-or-less
248480,California Chicken Dip,"[rotisserie cooked chicken shredded, red onion...","[1 rotisserie-cooked chicken, shredded ,...",[Put all ingredients in a large bowl &amp; mix...,30.0,"[15-minutes-or-less, time-to-make, course, pre...",1,57 g,[],"[dietary, low-carb]",[appetizers],[],[],"[chicken, onion, chili, cilantro mint, tomato,...",15-minutes-or-less
14906,Caramel Walnut Fudge Bars,"[cocoa powder, light brown sugar, vanilla, bro...","[5 tablespoons butter, 1/2 cup cocoa ...",[Make base: Line bottom of greased 13 X 9 inch...,1.0,"[60-minutes-or-less, time-to-make, course, mai...",1,1247 g,"[cookies-and-brownies, bar-cookies]","[dietary, low-sodium, kid-friendly, heirloom-h...",[desserts],"[christmas, thanksgiving, independence-day, ne...",[],"[chocolate, brown sugar, vanilla, brown sugar,...",60-minutes-or-less
431576,Creamy Curried Potatoes,"[medium potatoes, onion, curry powder, heavy c...","[5 medium potatoes, 1/2 medium onion,...",[Peel your potatoes and cut each into 6 pieces...,6.0,"[30-minutes-or-less, time-to-make, course, pre...",1,214 g,[],"[dietary, low-protein, low-sodium]",[side-dishes],[],[],"[potato, onion, curry powder, heavy cream]",30-minutes-or-less
208705,Cornbread - Healthy,"[cornmeal, baking soda, baking powder, butterm...","[2 1/2 cups cornmeal, 1 teaspoon salt...","[Heat oven to 425°F., Mix dry ingredients., Mi...",8.0,"[60-minutes-or-less, time-to-make, course, pre...",1,112 g,"[breads, quick-breads]","[dietary, low-protein, low-carb, low-sodium, l...",[],[],[],"[cornmeal, baking soda, baking powder, milk]",60-minutes-or-less
205662,Shrimp Cheese Fondue,"[cream shrimp soup, swiss cheese shredded, mil...","[1 (10 1/4 ounce) can cream of shrimp soup,...","[Combine all ingredients in a fondue pot., Hea...",1.0,"[30-minutes-or-less, time-to-make, main-ingred...",1,646 g,[],"[dietary, high-protein, low-carb, high-calcium]",[],[],[],"[shrimp, cheese, milk, garlic, shrimp]",30-minutes-o

## On enlève les recettes qui ont le même nom ET les mêmes ingrédients

In [225]:
df_filtered["ingredients_sorted"] = df_filtered["ingredients_clean"].apply(sorted)

In [233]:
df_filtered.reset_index(inplace=True)

In [234]:
del df_filtered["index"]

In [226]:
df_filtered["ingredients_tuple"] = df_filtered["ingredients_sorted"].apply(tuple)

df_filtered = df_filtered.drop_duplicates(
    subset=["name", "ingredients_tuple"]
)

In [227]:
df_filtered = df_filtered.drop(columns="ingredients_tuple")

In [228]:
len(df_filtered)

75198

# 75198 recettes !

In [230]:
df_filtered.head()

,name,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size,type_dish,type_diet,type_meal,type_occasion,type_origin,ingredients_clean,Time_to_make,ingredients_sorted
13,Crepes for Two,"[milk, vegetable oil, powdered sugar, orange m...","[1/2 cup flour, 3/4 cup milk, 1 ...","[Combine flour, milk, eggs, and oil and salt.,...",2.0,"[15-minutes-or-less, time-to-make, course, cui...",1,243 g,[pancakes-and-waffles],[dietary],[breakfast],[],"[french, european]","[milk, cooking oil, powdered sugar, orange, sp...",15-minutes-or-less,"[cooking oil, milk, orange, powdered sugar, sp..."
17,Peanut Butter and Banana Burrito,"[flour tortilla, peanut butter, banana]","[1 flour tortilla, 1 -2 tablespoon ...","[Heat tortilla over open flame of burner, till...",1.0,"[15-minutes-or-less, time-to-make, course, mai...",1,105 g,[],"[dietary, kid-friendly, inexpensive]","[brunch, breakfast, lunch]",[brunch],"[north-american, american, southern-united-sta...","[flour, peanut butter, banana]",15-minutes-or-less,"[banana, flour, peanut butter]"
25,Oil and Vinegar Salad Dressing,"[olive oil, white wine vinegar, dry mustard]","[3/4 cup olive oil, 5 tablespoons whi...",[Combine all ingredients in a jar and shake or...,1.0,"[15-minutes-or-less, time-to-make, course, pre...",1,169 g,"[salads, salad-dressings]",[dietary],[],[],[],"[cooking oil, vinegar, mustard powder]",15-minutes-or-less,"[cooking oil, mustard powder, vinegar]"
40,Southwestern Avocado Salsa,"[avocados, tomatoes, white shoepeg corn, black...","[2 avocados, diced , 2 tomatoes, d...",[Combine shoepeg corn and black beans in a med...,4.0,"[15-minutes-or-less, time-to-make, course, mai...",1,326 g,[vegetables],"[dietary, low-sodium, vegetarian, low-choleste...",[appetizers],[],[],"[avocado, tomato, corn, black bean, green onio...",15-minutes-or-less,"[avocado, black bean, cilantro mint, corn, gre..."
59,Chickpea Salad,"[chickpeas, celery, red onion, parsley, lemon ...","[16 ounces chickpeas, drained and rinsed ...","[Mix all ingredients together., Allow five min...",6.0,"[15-minutes-or-less, time-to-make, course, mai...",1,99 g,"[salads, vegetables]","[dietary, vegetarian, pasta-rice-and-grains, k...",[side-dishes],[],"[asian, middle-eastern]","[chickpea, celery, onion, parsley, lemon juice...",15-minutes-or-less,"[celery, chickpea, cooking oil, lemon juice, o..."


In [231]:
df_filtered = df_filtered.drop(columns=["tags","ingredients_sorted"])

In [235]:
df_filtered.head()

,name,ingredients,ingredients_raw,steps,servings,persons,portion_size,type_dish,type_diet,type_meal,type_occasion,type_origin,ingredients_clean,Time_to_make
0,Crepes for Two,"[milk, vegetable oil, powdered sugar, orange m...","[1/2 cup flour, 3/4 cup milk, 1 ...","[Combine flour, milk, eggs, and oil and salt.,...",2.0,1,243 g,[pancakes-and-waffles],[dietary],[breakfast],[],"[french, european]","[milk, cooking oil, powdered sugar, orange, sp...",15-minutes-or-less
1,Peanut Butter and Banana Burrito,"[flour tortilla, peanut butter, banana]","[1 flour tortilla, 1 -2 tablespoon ...","[Heat tortilla over open flame of burner, till...",1.0,1,105 g,[],"[dietary, kid-friendly, inexpensive]","[brunch, breakfast, lunch]",[brunch],"[north-american, american, southern-united-sta...","[flour, peanut butter, banana]",15-minutes-or-less
2,Oil and Vinegar Salad Dressing,"[olive oil, white wine vinegar, dry mustard]","[3/4 cup olive oil, 5 tablespoons whi...",[Combine all ingredients in a jar and shake or...,1.0,1,169 g,"[salads, salad-dressings]",[dietary],[],[],[],"[cooking oil, vinegar, mustard powder]",15-minutes-or-less
3,Southwestern Avocado Salsa,"[avocados, tomatoes, white shoepeg corn, black...","[2 avocados, diced , 2 tomatoes, d...",[Combine shoepeg corn and black beans in a med...,4.0,1,326 g,[vegetables],"[dietary, low-sodium, vegetarian, low-choleste...",[appetizers],[],[],"[avocado, tomato, corn, black bean, green onio...",15-minutes-or-less
4,Chickpea Salad,"[chickpeas, celery, red onion, parsley, lemon ...","[16 ounces chickpeas, drained and rinsed ...","[Mix all ingredients together., Allow five min...",6.0,1,99 g,"[salads, vegetables]","[dietary, vegetarian, pasta-rice-and-grains, k...",[side-dishes],[],"[asian, middle-eastern]","[chickpea, celery, onion, parsley, lemon juice...",15-minutes-or-less


In [241]:
df_filtered.to_csv("../../raw_data/recipe_final.csv", index=False)

In [240]:
df_filtered.drop(columns="ingredients_clean")

,name,ingredients,ingredients_raw,steps,servings,persons,portion_size,type_dish,type_diet,type_meal,type_occasion,type_origin,Time_to_make
0,Crepes for Two,"[milk, cooking oil, powdered sugar, orange, sp...","[1/2 cup flour, 3/4 cup milk, 1 ...","[Combine flour, milk, eggs, and oil and salt.,...",2.0,1,243 g,[pancakes-and-waffles],[dietary],[breakfast],[],"[french, european]",15-minutes-or-less
1,Peanut Butter and Banana Burrito,"[flour, peanut butter, banana]","[1 flour tortilla, 1 -2 tablespoon ...","[Heat tortilla over open flame of burner, till...",1.0,1,105 g,[],"[dietary, kid-friendly, inexpensive]","[brunch, breakfast, lunch]",[brunch],"[north-american, american, southern-united-sta...",15-minutes-or-less
2,Oil and Vinegar Salad Dressing,"[cooking oil, vinegar, mustard powder]","[3/4 cup olive oil, 5 tablespoons whi...",[Combine all ingredients in a jar and shake or...,1.0,1,169 g,"[salads, salad-dressings]",[dietary],[],[],[],15-minutes-or-less
3,Southwestern Avocado Salsa,"[avocado, tomato, corn, black bean, green onio...","[2 avocados, diced , 2 tomatoes, d...",[Combine shoepeg corn and black beans in a med...,4.0,1,326 g,[vegetables],"[dietary, low-sodium, vegetarian, low-choleste...",[appetizers],[],[],15-minutes-or-less
4,Chickpea Salad,"[chickpea, celery, onion, parsley, lemon juice...","[16 ounces chickpeas, drained and rinsed ...","[Mix all ingredients together., Allow five min...",6.0,1,99 g,"[salads, vegetables]","[dietary, vegetarian, pasta-rice-and-grains, k...",[side-dishes],[],"[asian, middle-eastern]",15-minutes-or-less
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75193,Glazed Snap Peas,"[pea, bell pepper, pork]",[2 (24 ounce) packages frozen sugar snap p...,[Cook peas according to package directions; dr...,10.0,1,150 g,[vegetables],"[dietary, kid-friendly]","[dinner-party, side-dishes]","[dinner-party, christmas, easter]","[north-american, american]",30-minutes-or-less
75194,Oriental Cabbage Salad,"[cabbage, green onion, sesame oil, vinegar, se...","[1/2 small green cabbage, 3 scallion...","[Combine cabbage, scallions, sesame oil and vi...",4.0,1,111 g,"[salads, vegetables]",[],[side-dishes],[],[],15-minutes-or-less
75195,Sherried Chicken,"[chicken, onion, alcohol]","[4 boneless skinless chicken breasts, 1 ...","[Preheat oven to 350., Spray a baking dish, I ...",4.0,1,175 g,[],"[dietary, low-carb, low-sodium, healthy, low-f...",[main-dish],[],[north-american],30-minutes-or-less
75196,Terriyaki Sauce,"[soy, water, ginger, garlic, brown sugar, corn...","[1/4 cup soy sauce, 1 cup water, 1/2 ...",[1. Mix all but cornstarch and 1/4c water in a...,1.0,1,476 g,"[sauces, savory-sauces, sweet-sauces]","[dietary, low-protein, low-cholesterol, health...",[],[],"[asian, chinese]",15-minutes-or-less


In [239]:
df_filtered.head()

,name,ingredients,ingredients_raw,steps,servings,persons,portion_size,type_dish,type_diet,type_meal,type_occasion,type_origin,ingredients_clean,Time_to_make
0,Crepes for Two,"[milk, cooking oil, powdered sugar, orange, sp...","[1/2 cup flour, 3/4 cup milk, 1 ...","[Combine flour, milk, eggs, and oil and salt.,...",2.0,1,243 g,[pancakes-and-waffles],[dietary],[breakfast],[],"[french, european]","[milk, cooking oil, powdered sugar, orange, sp...",15-minutes-or-less
1,Peanut Butter and Banana Burrito,"[flour, peanut butter, banana]","[1 flour tortilla, 1 -2 tablespoon ...","[Heat tortilla over open flame of burner, till...",1.0,1,105 g,[],"[dietary, kid-friendly, inexpensive]","[brunch, breakfast, lunch]",[brunch],"[north-american, american, southern-united-sta...","[flour, peanut butter, banana]",15-minutes-or-less
2,Oil and Vinegar Salad Dressing,"[cooking oil, vinegar, mustard powder]","[3/4 cup olive oil, 5 tablespoons whi...",[Combine all ingredients in a jar and shake or...,1.0,1,169 g,"[salads, salad-dressings]",[dietary],[],[],[],"[cooking oil, vinegar, mustard powder]",15-minutes-or-less
3,Southwestern Avocado Salsa,"[avocado, tomato, corn, black bean, green onio...","[2 avocados, diced , 2 tomatoes, d...",[Combine shoepeg corn and black beans in a med...,4.0,1,326 g,[vegetables],"[dietary, low-sodium, vegetarian, low-choleste...",[appetizers],[],[],"[avocado, tomato, corn, black bean, green onio...",15-minutes-or-less
4,Chickpea Salad,"[chickpea, celery, onion, parsley, lemon juice...","[16 ounces chickpeas, drained and rinsed ...","[Mix all ingredients together., Allow five min...",6.0,1,99 g,"[salads, vegetables]","[dietary, vegetarian, pasta-rice-and-grains, k...",[side-dishes],[],"[asian, middle-eastern]","[chickpea, celery, onion, parsley, lemon juice...",15-minutes-or-less
